# Testing Different Initialization Methods for APD Components

This notebook tests different ways to initialize A and B matrices to satisfy A @ B = W_original exactly.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Add the project root to the path
sys.path.append('.')

from spd.configs import Config
from spd.experiments.resid_mlp.models import ResidualMLP
from spd.models.component_model import ComponentModel
from spd.models.components import EmbeddingComponent, LinearComponent
from spd.utils import get_device, load_config, set_seed

device = get_device()
print(f"Using device: {device}")

## Load ResidualMLP Model and Config

In [ ]:
# Load a simple ResidualMLP config (1 layer)
config = load_config("spd/experiments/resid_mlp/resid_mlp_config.yaml", config_model=Config)

# For testing, use a smaller m to make it easier to see
config = config.model_copy(update={"m": 50, "wandb_project": None})

print(f"Config m: {config.m}")
print(f"Target patterns: {config.target_module_patterns}")

# Load the pretrained ResidualMLP model
print(f"Loading model from: {config.pretrained_model_path}")
target_model, target_model_train_config, label_coeffs = ResidualMLP.from_pretrained(
    config.pretrained_model_path
)
target_model = target_model.to(device)
target_model.eval()

print(f"Model: {target_model}")
print(f"Model config: {target_model.config}")

## Create ComponentModel and Extract Target Weights

In [ ]:
# Create ComponentModel
comp_model = ComponentModel(
    base_model=target_model,
    target_module_patterns=config.target_module_patterns,
    m=config.m,
    n_gate_hidden_neurons=config.n_gate_hidden_neurons,
    pretrained_model_output_attr=config.pretrained_model_output_attr,
    gate_type=config.gate_type,
)
comp_model.to(device)

# Get components
components = {
    k.removeprefix("components.").replace("-", "."): v 
    for k, v in comp_model.components.items()
}

print(f"Components: {list(components.keys())}")

# Print shapes and ranks
for name, component in components.items():
    target_weight = comp_model.model.get_parameter(name + ".weight")
    rank = torch.linalg.matrix_rank(target_weight).item()
    print(f"{name}: weight shape {target_weight.shape}, rank {rank}, m={config.m}")
    print(f"  A shape: {component.A.shape}, B shape: {component.B.shape}")

## Define Initialization Methods

In [ ]:
def init_As_and_Bs_min_norm_(
    model: ComponentModel, components: dict[str, LinearComponent | EmbeddingComponent]
) -> None:
    """Random A, minimum norm B solution."""
    for param_name, component in components.items():
        target_weight = model.model.get_parameter(param_name + ".weight")
        if isinstance(component, EmbeddingComponent):
            target_weight = target_weight.T
        
        # Random A 
        component.A.data = torch.randn_like(component.A.data)
        
        # Minimum norm solution: B = A^+ @ W where A^+ is pseudoinverse
        component.B.data = torch.linalg.pinv(component.A) @ target_weight


def init_As_and_Bs_svd_padded_(
    model: ComponentModel, components: dict[str, LinearComponent | EmbeddingComponent]  
) -> None:
    """SVD initialization, pad extra components with zeros."""
    for param_name, component in components.items():
        target_weight = model.model.get_parameter(param_name + ".weight")
        if isinstance(component, EmbeddingComponent):
            target_weight = target_weight.T
        
        # SVD
        U, S, Vt = torch.svd(target_weight)
        rank = torch.sum(S > 1e-6).item()  # Numerical rank
        m = component.A.shape[1]
        
        # Use first `rank` components for reconstruction, rest are zeros
        component.A.data.zero_()
        component.B.data.zero_()
        
        if rank > 0:
            # Fill first `rank` components with SVD
            use_rank = min(rank, m)
            sqrt_S = torch.sqrt(S[:use_rank])
            component.A.data[:, :use_rank] = Vt[:use_rank, :].T @ torch.diag(sqrt_S)
            component.B.data[:use_rank, :] = torch.diag(sqrt_S) @ U[:, :use_rank].T


def init_As_and_Bs_qr_(
    model: ComponentModel, components: dict[str, LinearComponent | EmbeddingComponent]
) -> None:
    """Initialize using QR decomposition."""
    for param_name, component in components.items():
        target_weight = model.model.get_parameter(param_name + ".weight")
        if isinstance(component, EmbeddingComponent):
            target_weight = target_weight.T
        
        m = component.A.shape[1]
        d_in = component.A.shape[0]
        
        # Random A with orthonormal columns
        A_random = torch.randn(d_in, m, device=component.A.device)
        Q, R = torch.qr(A_random)
        component.A.data = Q
        
        # Solve for B: A @ B = W_target
        component.B.data = torch.linalg.lstsq(component.A, target_weight).solution


# Original method from the codebase
def init_As_and_Bs_original_(
    model: ComponentModel, components: dict[str, LinearComponent | EmbeddingComponent]
) -> None:
    """Original initialization from the codebase."""
    for param_name, component in components.items():
        A = component.A
        B = component.B
        target_weight = model.model.get_parameter(param_name + ".weight")
        if isinstance(component, EmbeddingComponent):
            target_weight = target_weight.T  # (d_out d_in)

        # Make A and B have unit norm in the d_in and d_out dimensions
        A.data[:] = torch.randn_like(A.data)
        B.data[:] = torch.randn_like(B.data)
        A.data[:] = A.data / A.data.norm(dim=-2, keepdim=True)
        B.data[:] = B.data / B.data.norm(dim=-1, keepdim=True)

        # Calculate inner products and scale B
        import einops
        m_norms = einops.einsum(A, B, target_weight, "d_in m, m d_out, d_out d_in -> m")
        B.data[:] = B.data * m_norms.unsqueeze(-1)

## Test Each Initialization Method

In [ ]:
def test_initialization_method(init_fn, method_name):
    """Test an initialization method and return reconstruction errors."""
    print(f"\n=== Testing {method_name} ===")
    
    # Reset the model
    comp_model_test = ComponentModel(
        base_model=target_model,
        target_module_patterns=config.target_module_patterns,
        m=config.m,
        n_gate_hidden_neurons=config.n_gate_hidden_neurons,
        pretrained_model_output_attr=config.pretrained_model_output_attr,
        gate_type=config.gate_type,
    )
    comp_model_test.to(device)
    
    components_test = {
        k.removeprefix("components.").replace("-", "."): v 
        for k, v in comp_model_test.components.items()
    }
    
    # Apply initialization
    set_seed(42)  # For reproducibility
    init_fn(comp_model_test, components_test)
    
    # Calculate reconstruction errors
    errors = {}
    for name, component in components_test.items():
        target_weight = comp_model_test.model.get_parameter(name + ".weight")
        if isinstance(component, EmbeddingComponent):
            target_weight = target_weight.T
        
        reconstructed = component.A @ component.B
        error = torch.norm(reconstructed - target_weight).item()
        rel_error = error / torch.norm(target_weight).item()
        
        errors[name] = {
            'absolute_error': error,
            'relative_error': rel_error,
            'target_norm': torch.norm(target_weight).item(),
            'reconstructed_norm': torch.norm(reconstructed).item()
        }
        
        print(f"{name}:")
        print(f"  Absolute error: {error:.2e}")
        print(f"  Relative error: {rel_error:.2e}")
        print(f"  Target norm: {torch.norm(target_weight).item():.2f}")
        print(f"  Reconstructed norm: {torch.norm(reconstructed).item():.2f}")
    
    return errors

# Test all methods
methods = [
    (init_As_and_Bs_original_, "Original (codebase)"),
    (init_As_and_Bs_min_norm_, "Minimum Norm"),
    (init_As_and_Bs_svd_padded_, "SVD + Padding"),
    (init_As_and_Bs_qr_, "QR Decomposition"),
]

all_results = {}
for init_fn, method_name in methods:
    try:
        all_results[method_name] = test_initialization_method(init_fn, method_name)
    except Exception as e:
        print(f"Error testing {method_name}: {e}")
        all_results[method_name] = None

## Compare Results

In [ ]:
# Create a summary comparison
print("\n" + "="*80)
print("SUMMARY COMPARISON")
print("="*80)

for method_name, results in all_results.items():
    if results is None:
        print(f"{method_name}: FAILED")
        continue
        
    print(f"\n{method_name}:")
    total_abs_error = sum(r['absolute_error'] for r in results.values())
    avg_rel_error = np.mean([r['relative_error'] for r in results.values()])
    max_rel_error = max(r['relative_error'] for r in results.values())
    
    print(f"  Total absolute error: {total_abs_error:.2e}")
    print(f"  Average relative error: {avg_rel_error:.2e}")
    print(f"  Max relative error: {max_rel_error:.2e}")
    
    # Check if it's essentially exact
    if max_rel_error < 1e-10:
        print("  ✓ EXACT reconstruction (within numerical precision)")
    elif max_rel_error < 1e-6:
        print("  ✓ Very good reconstruction")
    elif max_rel_error < 1e-3:
        print("  ~ Reasonable reconstruction")
    else:
        print("  ✗ Poor reconstruction")

## Visualize Component Usage

In [ ]:
# For SVD method, let's see how many components are actually used
def analyze_component_usage(init_fn, method_name):
    comp_model_test = ComponentModel(
        base_model=target_model,
        target_module_patterns=config.target_module_patterns,
        m=config.m,
        n_gate_hidden_neurons=config.n_gate_hidden_neurons,
        pretrained_model_output_attr=config.pretrained_model_output_attr,
        gate_type=config.gate_type,
    )
    comp_model_test.to(device)
    
    components_test = {
        k.removeprefix("components.").replace("-", "."): v 
        for k, v in comp_model_test.components.items()
    }
    
    set_seed(42)
    init_fn(comp_model_test, components_test)
    
    fig, axes = plt.subplots(2, len(components_test), figsize=(4*len(components_test), 8))
    if len(components_test) == 1:
        axes = axes.reshape(2, 1)
    
    for i, (name, component) in enumerate(components_test.items()):
        # A matrix norms
        A_norms = torch.norm(component.A, dim=0).cpu().numpy()
        axes[0, i].bar(range(len(A_norms)), A_norms)
        axes[0, i].set_title(f"{name} - A column norms")
        axes[0, i].set_ylabel("Norm")
        
        # B matrix norms
        B_norms = torch.norm(component.B, dim=1).cpu().numpy()
        axes[1, i].bar(range(len(B_norms)), B_norms)
        axes[1, i].set_title(f"{name} - B row norms")
        axes[1, i].set_ylabel("Norm")
        axes[1, i].set_xlabel("Component index")
        
        # Count non-zero components
        nonzero_A = torch.sum(A_norms > 1e-6).item()
        nonzero_B = torch.sum(B_norms > 1e-6).item()
        print(f"{name}: {nonzero_A} non-zero A columns, {nonzero_B} non-zero B rows")
    
    plt.suptitle(f"Component Usage - {method_name}")
    plt.tight_layout()
    plt.show()

# Analyze SVD method
print("\nAnalyzing component usage for SVD method:")
analyze_component_usage(init_As_and_Bs_svd_padded_, "SVD + Padding")

## Conclusion

This notebook demonstrates different initialization methods for ensuring A @ B = W_original:

1. **Minimum Norm**: Random A, solve for minimum norm B
2. **SVD + Padding**: Use SVD for exact rank components, pad rest with zeros
3. **QR Decomposition**: Orthogonal A, solve for B
4. **Original**: Current codebase method (approximate)

The exact methods should give reconstruction errors near machine precision (~1e-15), while the original method gives an approximate initialization.